# World Commodities (Yahoo Finance)

In [ ]:
from typing import cast
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf
import mgplot as mg

In [2]:
CHART_DIR = "./CHARTS/Commodities/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False

In [3]:
# --- Fetch WTI & Brent ---
tickers = {"CL=F": "WTI (US)", "BZ=F": "Brent (Europe)"}
frames = {}
for ticker, label in tickers.items():
    df = yf.download(ticker, start="2025-12-01", auto_adjust=True, progress=False)
    if df is not None:
        frames[label] = df["Close"].squeeze()

wti_brent = pd.DataFrame(frames)
wti_brent.index = cast(pd.DatetimeIndex, wti_brent.index).to_period("D")

print(f"Date range: {wti_brent.index[0]} to {wti_brent.index[-1]}")
for col in wti_brent.columns:
    print(f"  {col:20s}  min={wti_brent[col].min():.2f}  max={wti_brent[col].max():.2f}")

Date range: 2025-12-01 to 2026-04-10
  WTI (US)              min=55.27  max=112.95
  Brent (Europe)        min=58.92  max=118.35


In [4]:
# --- Chart: WTI vs Brent ---
data_to = wti_brent.dropna().index[-1].strftime("%-d-%b-%Y")
mg.line_plot_finalise(
    wti_brent,
    title="Crude Oil Prices: WTI vs Brent",
    ylabel="USD per barrel",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter=f"NYMEX (WTI) and ICE (Brent) daily prices. Data to {data_to}.",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [ ]:
# --- Fetch WTI & Brent forward curves (dated contracts) ---
# Yahoo exposes each dated futures contract as <root><month><YY>.NYM
# Month codes (CME standard): F G H J K M N Q U V X Z → Jan..Dec
MONTH_CODES = "FGHJKMNQUVXZ"
N_FORWARD_MONTHS = 18


def _contract_months(n_months: int) -> list[pd.Period]:
    """Return a list of contract expiry months starting from next month."""
    start = pd.Period(pd.Timestamp.today(), freq="M") + 1
    return [start + i for i in range(n_months)]


def _symbol(root: str, period: pd.Period) -> str:
    code = MONTH_CODES[period.month - 1]
    yy = period.year % 100
    return f"{root}{code}{yy:02d}.NYM"


def fetch_forward_curve(root: str, n_months: int) -> pd.DataFrame:
    """Fetch latest close + trade date for each dated contract.

    Returns a DataFrame indexed by contract expiry month with columns
    'price' and 'date'. Missing/expired contracts are skipped.
    """
    records = {}
    for m in _contract_months(n_months):
        sym = _symbol(root, m)
        try:
            hist = yf.Ticker(sym).history(period="5d")
        except Exception:
            continue
        if hist is None or len(hist) == 0:
            continue
        records[m] = {
            "price": float(hist["Close"].iloc[-1]),
            "date": hist.index[-1].tz_localize(None).normalize(),
        }
    return pd.DataFrame.from_dict(records, orient="index").sort_index()


def build_forward_curves(n_months: int) -> tuple[pd.DataFrame, pd.Timestamp]:
    """Return (price DataFrame, latest trade date across all contracts)."""
    wti = fetch_forward_curve("CL", n_months)
    brent = fetch_forward_curve("BZ", n_months)
    prices = pd.DataFrame({
        "WTI (US)": wti["price"],
        "Brent (Europe)": brent["price"],
    })
    prices.index = pd.PeriodIndex(prices.index, freq="M")
    latest = max(wti["date"].max(), brent["date"].max())
    return prices, latest


def summarise_curves(df: pd.DataFrame, as_of: pd.Timestamp) -> None:
    print(f"As of {as_of.date()}")
    print(f"Contracts: {df.index[0]} to {df.index[-1]}")
    for col in df.columns:
        valid = df[col].dropna()
        print(f"  {col:20s}  n={len(valid)}  front={valid.iloc[0]:.2f}  back={valid.iloc[-1]:.2f}")


forward, as_of = build_forward_curves(N_FORWARD_MONTHS)
summarise_curves(forward, as_of)

In [ ]:
# --- Chart: WTI & Brent forward curves ---
def plot_forward_curves(df: pd.DataFrame, as_of: pd.Timestamp) -> None:
    as_of_str = as_of.strftime("%-d-%b-%Y")
    mg.line_plot_finalise(
        df,
        title="Crude Oil Forward Curves: WTI and Brent",
        ylabel="USD per barrel",
        xlabel="Contract month",
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        marker="o",
        markersize=4,
        lfooter=(
            f"Latest settle price for each dated contract (delivery within month). "
            f"As of {as_of_str}."
        ),
        rfooter="Source: Yahoo Finance (NYMEX CL, ICE BZ)",
        show=SHOW,
    )


plot_forward_curves(forward, as_of)

In [5]:
# --- Fetch DME Oman & Dubai from Oilprice.com (cached) ---
CACHE_DIR = Path("./CACHE_SOURCE/")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OILPRICE_BLENDS = {
    "Oman (Middle East)": {"blend_id": 48, "cache": "dme_oman.csv"},
    "Dubai (Middle East)": {"blend_id": 144, "cache": "dubai.csv"},
}


def _fetch_oilprice(blend_id: int, period: int) -> pd.DataFrame:
    """Fetch crude oil prices from Oilprice.com API."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36"
        ),
        "X-Requested-With": "XMLHttpRequest",
        "Referer": "https://oilprice.com/oil-price-charts/",
    }
    csrf = requests.get(
        "https://oilprice.com/ajax/csrf", headers=headers, timeout=15
    ).json()
    resp = requests.post(
        "https://oilprice.com/freewidgets/json_get_oilprices",
        headers=headers,
        data={"blend_id": blend_id, "period": period, csrf["name"]: csrf["hash"]},
        timeout=15,
    )
    resp.raise_for_status()
    records = resp.json()["prices"]
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["time"], unit="s").dt.normalize()
    df["price"] = df["price"].astype(float)
    # Drop stale artifact entries (timestamps far out of sequence)
    df = df.sort_values("date").reset_index(drop=True)
    if len(df) > 1:
        df = df[df["date"] >= df["date"].iloc[-1] - pd.Timedelta(days=400)]
    return df[["date", "price"]]


oilprice_series = {}
for name, cfg in OILPRICE_BLENDS.items():
    cache_file = CACHE_DIR / cfg["cache"]
    if cache_file.exists():
        cached = pd.read_csv(cache_file, parse_dates=["date"])
        fresh = _fetch_oilprice(cfg["blend_id"], period=6)
        combined = pd.concat([cached, fresh]).drop_duplicates(subset="date", keep="last")
        df = combined.sort_values("date").reset_index(drop=True)
        print(f"{name}: updated cache ({len(cached)} -> {len(df)} rows)")
    else:
        df = _fetch_oilprice(cfg["blend_id"], period=5)
        print(f"{name}: initial cache ({len(df)} rows)")
    df.to_csv(cache_file, index=False)
    oilprice_series[name] = pd.Series(
        df["price"].values,
        index=pd.PeriodIndex(df["date"], freq="D"),
        name=name,
    )

Oman (Middle East): updated cache (202 -> 202 rows)


Dubai (Middle East): updated cache (214 -> 214 rows)


In [6]:
# --- Combine all benchmarks ---
prices = wti_brent.copy()
for s in oilprice_series.values():
    prices = prices.join(s, how="outer")
prices = prices.sort_index()
prices = prices.loc[prices.index >= pd.Period("2025-12-01", freq="D")]

print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
for col in prices.columns:
    valid = prices[col].dropna()
    print(f"  {col:20s}  min={valid.min():.2f}  max={valid.max():.2f}  n={len(valid)}")

Date range: 2025-12-01 to 2026-04-10
  WTI (US)              min=55.27  max=112.95  n=90
  Brent (Europe)        min=58.92  max=118.35  n=90
  Oman (Middle East)    min=58.36  max=162.72  n=69
  Dubai (Middle East)   min=58.35  max=137.82  n=68


In [7]:
# --- Chart: All Crude Oil Benchmarks ---
last_dates = {col: prices[col].dropna().index[-1] for col in prices.columns}
lfooter = "Last: " + ", ".join(f"{col.split(' ')[0]} {d}" for col, d in last_dates.items())

mg.line_plot_finalise(
    prices,
    title="Crude Oil Benchmarks",
    ylabel="USD per barrel",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter=lfooter,
    rfooter="Source: Yahoo Finance / Oilprice.com",
    show=SHOW,
)

print(f"\nChart saved to {CHART_DIR}")


Chart saved to ./CHARTS/Commodities/


In [8]:
# --- Fetch and chart single commodities ---
def fetch_and_chart(ticker: str, name: str, ylabel: str, start: str = "2024-03-20"):
    """Fetch a single commodity from Yahoo Finance and chart it."""
    raw = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    series = raw["Close"].squeeze().dropna()
    if len(series) == 0:
        print(f"{name} ({ticker}): no data available, skipping")
        return None
    series.index = cast(pd.DatetimeIndex, series.index).to_period("D")
    data_to = series.index[-1].strftime("%-d-%b-%Y")
    print(f"{name}: {series.index[0]} to {data_to}  min={series.min():.2f}  max={series.max():.2f}")
    mg.line_plot_finalise(
        series,
        title=f"{name} Price",
        ylabel=ylabel,
        xlabel=None,
        annotate=True,
        lfooter=f"Daily price. Data to {data_to}.",
        rfooter="Source: Yahoo Finance",
        show=SHOW,
    )
    return series


for ticker, name, ylabel in [
    ("GC=F", "Gold", "USD per troy ounce"),
    ("SI=F", "Silver", "USD per troy ounce"),
    ("PL=F", "Platinum", "USD per troy ounce"),
    ("PA=F", "Palladium", "USD per troy ounce"),
    ("HG=F", "Copper", "USD per pound"),
    ("TIO=F", "Iron Ore", "USD per tonne"),
]:
    fetch_and_chart(ticker, name, ylabel)

Gold: 2024-03-20 to 10-Apr-2026  min=2157.90  max=5318.40


Silver: 2024-03-20 to 10-Apr-2026  min=24.48  max=115.08


Platinum: 2024-03-20 to 10-Apr-2026  min=894.00  max=2852.40


Palladium: 2024-03-20 to 10-Apr-2026  min=822.80  max=2169.90


Copper: 2024-03-20 to 10-Apr-2026  min=3.94  max=6.18


Iron Ore: 2024-03-20 to 9-Apr-2026  min=91.28  max=119.56


In [9]:
# --- Fetch natural gas data ---
gas_tickers = {
    "NG=F": "Henry Hub (US)",
    "TTF=F": "TTF (Europe)",
    "JKM=F": "JKM (Asia)",
}
gas_frames = {}
for ticker, label in gas_tickers.items():
    df = yf.download(ticker, start="2025-12-01", auto_adjust=True, progress=False)
    if df is not None:
        s = df["Close"].squeeze().dropna()
        s.index = cast(pd.DatetimeIndex, s.index).to_period("D")
        gas_frames[label] = s

# Fetch EUR/USD rate to convert TTF from EUR/MWh to USD/MMBtu
fx_df = yf.download("EURUSD=X", start="2025-12-01", auto_adjust=True, progress=False)
if fx_df is not None:
    eurusd = fx_df["Close"].squeeze().dropna()
    eurusd.index = cast(pd.DatetimeIndex, eurusd.index).to_period("D")

gas_prices = pd.DataFrame(gas_frames).dropna()

# Convert TTF: EUR/MWh -> USD/MMBtu (1 MWh = 3.412 MMBtu)
MMBTU_PER_MWH = 3.412
ttf_aligned = eurusd.reindex(gas_prices.index, method="ffill")
gas_prices["TTF (Europe)"] = gas_prices["TTF (Europe)"] * ttf_aligned / MMBTU_PER_MWH

print(f"Natural Gas date range: {gas_prices.index[0]} to {gas_prices.index[-1]}")
for col in gas_prices.columns:
    print(f"  {col:20s}  min={gas_prices[col].min():.3f}  max={gas_prices[col].max():.3f}")

Natural Gas date range: 2025-12-01 to 2026-04-09
  Henry Hub (US)        min=2.670  max=7.460
  TTF (Europe)          min=9.066  max=20.783
  JKM (Asia)            min=9.455  max=22.350


In [10]:
# --- Chart 4: Natural Gas Benchmarks ---
data_to = gas_prices.dropna().index[-1].strftime("%-d-%b-%Y")
mg.line_plot_finalise(
    gas_prices,
    title="Natural Gas Benchmarks",
    ylabel="USD per MMBtu",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    axvline={
        "x": pd.Period("2026-02-28", freq="D").ordinal,
        "color": "grey",
        "linestyle": "--",
        "linewidth": 1,
        "label": "Hormuz closure",
    },
    lfooter=f"TTF converted from EUR/MWh using daily EUR/USD rate. Data to {data_to}.",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [11]:
# --- Fetch grains data (last 2 years) ---
grain_tickers = {"ZW=F": "Wheat", "ZC=F": "Corn"}
grain_frames = {}
for ticker, label in grain_tickers.items():
    df = yf.download(ticker, start="2024-03-20", auto_adjust=True, progress=False)
    if df is not None:
        s = df["Close"].squeeze().dropna()
        s.index = cast(pd.DatetimeIndex, s.index).to_period("D")
        grain_frames[label] = s

grains = pd.DataFrame(grain_frames).dropna()

print(f"Grains date range: {grains.index[0]} to {grains.index[-1]}")
for col in grains.columns:
    print(f"  {col:6s}  min={grains[col].min():.2f}  max={grains[col].max():.2f}")

Grains date range: 2024-03-20 to 2026-04-10
  Wheat   min=495.00  max=700.25
  Corn    min=362.00  max=502.00


In [12]:
# --- Chart: Grain Prices ---
data_to = grains.dropna().index[-1].strftime("%-d-%b-%Y")
mg.line_plot_finalise(
    grains,
    title="Grain Prices: Wheat and Corn",
    ylabel="USD cents per bushel",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    lfooter=f"CBOT daily prices. Data to {data_to}.",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

In [13]:
# --- Softs, grains, and other commodities ---
for ticker, name, ylabel in [
    ("ZS=F", "Soybeans", "USD cents per bushel"),
    ("KC=F", "Coffee", "USD cents per pound"),
    ("SB=F", "Sugar", "USD cents per pound"),
    ("CC=F", "Cocoa", "USD per tonne"),
    ("CT=F", "Cotton", "USD cents per pound"),
    ("UFV=F", "Urea", "USD per tonne"),
]:
    fetch_and_chart(ticker, name, ylabel)

Soybeans: 2024-03-20 to 10-Apr-2026  min=938.75  max=1248.00


Coffee: 2024-03-20 to 10-Apr-2026  min=182.40  max=438.90


Sugar: 2024-03-20 to 10-Apr-2026  min=13.72  max=23.42


Cocoa: 2024-03-20 to 10-Apr-2026  min=2798.00  max=12565.00


Cotton: 2024-03-20 to 9-Apr-2026  min=61.06  max=93.41


Urea: 2024-03-20 to 9-Apr-2026  min=283.00  max=708.00


In [14]:
# --- Fetch ASX indices ---
asx_tickers = {"^AORD": "All Ordinaries", "^AXJO": "S&P/ASX 200"}
asx_frames = {}
for ticker, label in asx_tickers.items():
    df = yf.download(ticker, start="2024-03-20", auto_adjust=True, progress=False)
    if df is not None:
        asx_frames[label] = df["Close"].squeeze().dropna()

asx = pd.DataFrame(asx_frames)
asx.index = cast(pd.DatetimeIndex, asx.index).to_period("D")

data_to = asx.dropna().index[-1].strftime("%-d-%b-%Y")
print(f"ASX date range: {asx.index[0]} to {data_to}")
for col in asx.columns:
    print(f"  {col:20s}  min={asx[col].min():.0f}  max={asx[col].max():.0f}")

mg.line_plot_finalise(
    asx,
    title="ASX Indices: All Ordinaries and S&P/ASX 200",
    ylabel="Index",
    xlabel=None,
    legend={"loc": "best", "fontsize": "x-small"},
    annotate=True,
    lfooter=f"Daily close. Data to {data_to}.",
    rfooter="Source: Yahoo Finance",
    show=SHOW,
)

ASX date range: 2024-03-20 to 9-Apr-2026
  All Ordinaries        min=7524  max=9436
  S&P/ASX 200           min=7343  max=9199


In [15]:
# --- ASX Small Ordinaries ---
fetch_and_chart("^AXSO", "S&P/ASX Small Ordinaries", "Index")

S&P/ASX Small Ordinaries: 2024-03-20 to 9-Apr-2026  min=2751.40  max=3992.50


Date
2024-03-20    3034.399902
2024-03-21    3092.800049
2024-03-22    3061.100098
2024-03-25    3072.300049
2024-03-26    3070.100098
                 ...     
2026-04-01    3416.899902
2026-04-02    3331.300049
2026-04-07    3373.199951
2026-04-08    3512.000000
2026-04-09    3478.800049
Freq: D, Name: ^AXSO, Length: 519, dtype: float64

In [16]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-04-11 10:30:16

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.12.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot  : 0.2.21
pandas  : 3.0.2
pathlib : 1.0.1
requests: 2.33.1
typing  : 3.10.0.0
yfinance: 1.2.0

Watermark: 2.6.0



In [17]:
print("Finished")

Finished
